# Test/Train Split

**Strategy:**
- **Test Base -- PS+**: All images from PS-labeled surveys. Uses PS labels where available, fills gaps alphabetically.
- **Test Base -- Duplicates**: Other labelers' labels on the same images (for comparison only)
- **Test Aug**: Additional surveys if needed
- **Test Total**: PS+ + Aug
- **Train**: Everything else

In [62]:
import pandas as pd
import numpy as np
import re

# Load data
df = pd.read_csv('../data/labels/labeled_surveys/random_sample/latest_irrigation_table.csv')
df = df[df['operator_initials'].isin(['DSB', 'JL', 'KL', 'MV', 'PS', 'AB'])].copy()

# Create image_id and survey_id
df['image_id'] = df['site_id'] + '_' + df['year'].astype(str) + '-' + df['month'].astype(str).str.zfill(2) + '-' + df['day'].astype(str).str.zfill(2)
df['survey_id'] = df['source_file'].apply(lambda x: re.search(r'(\d+-\d+)', str(x)).group(1) if pd.notna(x) and re.search(r'(\d+-\d+)', str(x)) else None)

print(f"Total labels: {len(df)}, Unique images: {df['image_id'].nunique()}")
print(f"\nLabels by operator:")
print(df['operator_initials'].value_counts().to_string())

# Helper
def in_surveys(source_file, surveys):
    return pd.notna(source_file) and any(s in str(source_file) for s in surveys)

Total labels: 2770, Unique images: 2350

Labels by operator:
operator_initials
KL     678
MV     623
DSB    588
JL     582
PS     248
AB      51


In [45]:
# =============================================================================
# CONFIGURATION - PS_SURVEYS
# =============================================================================
PS_SURVEYS = ['101-125', '775-799', '825-849', '975-999', '1025-1049']

In [46]:
# Select augmentation surveys
all_surveys = sorted(df['survey_id'].dropna().unique(), key=lambda x: int(x.split('-')[0]))
non_ps = [s for s in all_surveys if s not in PS_SURVEYS]

# Industrial candidates
print("\nNon-PS surveys with industrial irrigation:")
for s in non_ps:
    sdata = df[df['survey_id'] == s]
    ind = (sdata['percent_coverage_hc_industrial'] > 0).sum()
    if ind > 0:
        print(f"  {s}: {ind} industrial, {(sdata['irrigation'] >= 3).sum()} irrigated")


Non-PS surveys with industrial irrigation:
  51-75: 4 industrial, 18 irrigated
  176-200: 4 industrial, 27 irrigated
  275-299: 3 industrial, 13 irrigated
  325-349: 4 industrial, 18 irrigated
  375-399: 1 industrial, 8 irrigated
  400-424: 2 industrial, 11 irrigated
  449-473: 2 industrial, 11 irrigated
  474-498: 2 industrial, 27 irrigated
  550-574: 1 industrial, 27 irrigated
  700-724: 8 industrial, 32 irrigated
  850-874: 6 industrial, 17 irrigated
  875-899: 1 industrial, 13 irrigated


In [47]:
# =============================================================================
# CONFIGURATION - AUG_SURVEYS
# =============================================================================
AUG_SURVEYS = ['850-874', '875-899']  # Example additional surveys for test set

In [ ]:
# Mark PS surveys
df['is_ps_survey'] = df['source_file'].apply(lambda x: in_surveys(x, PS_SURVEYS))

# Select augmentation surveys
test_surveys = PS_SURVEYS + AUG_SURVEYS
df['is_test'] = df['source_file'].apply(lambda x: in_surveys(x, test_surveys))

# TEST BASE -- PS+: Deduplicate PS surveys (PS first, then alphabetical)
ps_data = df[df['is_ps_survey']].copy()
ps_data['priority'] = ps_data['operator_initials'].map({'PS': 0, 'DSB': 1, 'JL': 2, 'KL': 3, 'MV': 4, 'AB': 5})
test_ps_plus = ps_data.sort_values('priority').drop_duplicates('image_id', keep='first').drop(columns='priority')

# TEST BASE -- Duplicates: Remaining labels on same images
test_duplicates = ps_data[~ps_data.index.isin(test_ps_plus.index) & ps_data['image_id'].isin(test_ps_plus['image_id'])].copy()
if 'priority' in test_duplicates.columns:
    test_duplicates = test_duplicates.drop(columns='priority')

# TEST AUG
if AUG_SURVEYS:
    aug_data = df[df['source_file'].apply(lambda x: in_surveys(x, AUG_SURVEYS))].copy()
    aug_data['priority'] = aug_data['operator_initials'].map({'KL': 1, 'MV': 2, 'DSB': 3, 'JL': 4, 'AB': 5})
    test_aug = aug_data.sort_values('priority').drop_duplicates('image_id', keep='first').drop(columns='priority')
else:
    test_aug = pd.DataFrame(columns=df.columns)

# TEST TOTAL & TRAIN
test_total = pd.concat([test_ps_plus, test_aug], ignore_index=True)
train_data = df[~df['is_test']].copy()
train_data['priority'] = train_data['operator_initials'].map({'KL': 1, 'MV': 2, 'DSB': 3, 'JL': 4, 'AB': 5})
train_data = train_data.sort_values('priority').drop_duplicates('image_id', keep='first').drop(columns='priority')

# TOTAL
total = pd.concat([test_total, train_data], ignore_index=True)

# Summary
n_ps = (test_ps_plus['operator_initials'] == 'PS').sum()
print(f"Test Base -- PS+:        {len(test_ps_plus):4d} images ({n_ps} PS, {len(test_ps_plus)-n_ps} filled).    {len(test_ps_plus)/len(total):.2%} of total")
print(f"Test Base -- Duplicates: {len(test_duplicates):4d} labels")
print(f"Test Aug:                {len(test_aug):4d} images.                         {len(test_aug)/len(total):.2%} of total")
print(f"Test Total:              {len(test_total):4d} images.                       {len(test_total)/len(total):.2%} of total")
print(f"Train:                   {len(train_data):4d} images.                      {len(train_data)/len(total):.2%} of total")
print(f"Total:                   {len(total):4d} images")

Test Base -- PS+:         282 images (248 PS, 34 filled).    12.01% of total
Test Base -- Duplicates:  368 labels
Test Aug:                 109 images.                         4.64% of total
Test Total:               391 images.                       16.65% of total
Train:                   1958 images.                      83.35% of total
Total:                   2349 images


In [58]:
# Check label overlap in PS surveys
ps_data = df[df['is_ps_survey']].copy()
overlap = ps_data.groupby('image_id')['operator_initials'].apply(set).reset_index()
overlap['has_ps'] = overlap['operator_initials'].apply(lambda x: 'PS' in x)
overlap['has_other'] = overlap['operator_initials'].apply(lambda x: len(x - {'PS'}) > 0)

both = overlap[overlap['has_ps'] & overlap['has_other']]
only_ps = overlap[overlap['has_ps'] & ~overlap['has_other']]
only_other = overlap[~overlap['has_ps'] & overlap['has_other']]

print(f"Label overlap in PS surveys ({len(overlap)} unique images):")
print(f"  Both PS and others: {len(both)}")
print(f"  PS only:            {len(only_ps)}")
print(f"  Others only:        {len(only_other)}")

Label overlap in PS surveys (282 unique images):
  Both PS and others: 221
  PS only:            27
  Others only:        34


In [59]:
# Surveys in each split
print("TEST BASE -- PS+ surveys:", sorted(test_ps_plus['survey_id'].dropna().unique(), key=lambda x: int(x.split('-')[0])))
print("\nPS+ breakdown by labeler:")
print(test_ps_plus['operator_initials'].value_counts().to_string())
if AUG_SURVEYS:
    print(f"\nTEST AUG surveys: {sorted(AUG_SURVEYS, key=lambda x: int(x.split('-')[0]))}")
print(f"\nTRAIN surveys ({len(train_data['survey_id'].unique())}): {sorted(train_data['survey_id'].dropna().unique(), key=lambda x: int(x.split('-')[0]))}")

TEST BASE -- PS+ surveys: ['101-125', '775-799', '825-849', '975-999', '1025-1049']

PS+ breakdown by labeler:
operator_initials
PS     248
DSB     10
MV       9
KL       8
JL       7

TEST AUG surveys: ['850-874', '875-899']

TRAIN surveys (34): ['1-25', '26-50', '51-75', '126-150', '151-175', '176-200', '201-225', '225-249', '250-274', '275-299', '300-324', '325-349', '350-374', '375-399', '400-424', '425-449', '449-473', '474-498', '499-523', '525-549', '550-574', '575-599', '600-624', '625-649', '650-674', '675-699', '700-724', '725-749', '750-774', '800-824', '900-924', '925-949', '950-974', '1000-1024']


In [60]:
# Statistics
def stats(data, name):
    if len(data) == 0:
        return {'Split': name, 'Images': 0, 'Locations': 0, 'Irrigated': 0, '% Irr': 0, 'Locs w/ Irr': 0, 'Small-scale': 0, 'Industrial': 0}
    return {
        'Split': name,
        'Images': len(data),
        'Locations': data['site_id'].nunique(),
        'Irrigated': (data['irrigation'] >= 3).sum(),
        '% Irr': round((data['irrigation'] >= 3).mean() * 100, 1),
        'Locs w/ Irr': data[data['irrigation'] >= 3]['site_id'].nunique(),
        'Small-scale': (data['percent_coverage_hc_small-scale'] > 0).sum(),
        'Industrial': (data['percent_coverage_hc_industrial'] > 0).sum()
    }

splits = [('Test PS+', test_ps_plus), ('Test Duplicates', test_duplicates), ('Test Aug', test_aug), ('Test Total', test_total), ('Train', train_data)]
stats_df = pd.DataFrame([stats(d, n) for n, d in splits]).set_index('Split')
print(stats_df.to_string())

                 Images  Locations  Irrigated  % Irr  Locs w/ Irr  Small-scale  Industrial
Split                                                                                     
Test PS+            282        110         58   20.6           31           58           0
Test Duplicates     368        100         82   22.3           28           80           0
Test Aug            109         48         30   27.5           17           21           7
Test Total          391        158         88   22.5           48           79           7
Train              1958        781        549   28.0          269          514          31


In [61]:
# Assessment
t = stats_df.loc['Test Total']
print(f"Test set: {int(t['Images'])} images, {int(t['Locs w/ Irr'])} irrigated locations, {int(t['Industrial'])} industrial")
if t['Locs w/ Irr'] < 30:
    print("⚠️  < 30 irrigated locations")
if t['Industrial'] < 5:
    print("⚠️  < 5 industrial images")

Test set: 391 images, 48 irrigated locations, 7 industrial
